# Inference Intermediate Fusion

Notebook ini memuat checkpoint terbaik `best_intermediate_fusion.pt` dan menghasilkan prediksi berupa label serta probabilitas. Default checkpoint diambil dari `multimodal_citybranding/outputs`, dengan fallback ke folder `ouputmodel` jika diperlukan.


In [ ]:
# Jalankan jika dependency belum tersedia di kernel notebook ini.
# %pip install torch transformers pillow pandas tqdm


In [1]:
import json
import sys
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET

import pandas as pd
import torch
import torch.nn.functional as F

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "multimodal_citybranding").exists() else CWD.parent
PACKAGE_DIR = PROJECT_ROOT / "multimodal_citybranding"
DEFAULT_IMAGE_DIR = PROJECT_ROOT / "dataset" / "images_skema2"
DEFAULT_LABEL_FILE = PROJECT_ROOT / "dataset" / "labels_skema2.xlsx"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from multimodal_citybranding.main import resolve_device
from multimodal_citybranding.models import (
    IntermediateFusionClassifier,
    MultimodalFeatureExtractor,
)


def _column_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for ch in letters:
        index = index * 26 + ord(ch.upper()) - ord("A") + 1
    return index - 1


def read_xlsx_basic(path):
    path = Path(path)
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

    with ZipFile(path) as archive:
        shared_strings = []
        if "xl/sharedStrings.xml" in archive.namelist():
            root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            for item in root.findall("a:si", ns):
                shared_strings.append(
                    "".join(text.text or "" for text in item.findall(".//a:t", ns))
                )

        sheet_files = sorted(
            name
            for name in archive.namelist()
            if name.startswith("xl/worksheets/sheet") and name.endswith(".xml")
        )
        if not sheet_files:
            raise ValueError(f"Tidak ada worksheet di {path}")

        root = ET.fromstring(archive.read(sheet_files[0]))
        rows = []
        for row in root.findall(".//a:sheetData/a:row", ns):
            values = {}
            max_idx = -1
            for cell in row.findall("a:c", ns):
                idx = _column_index(cell.get("r", "A1"))
                max_idx = max(max_idx, idx)
                cell_type = cell.get("t")

                if cell_type == "inlineStr":
                    text_node = cell.find("a:is/a:t", ns)
                    value = "" if text_node is None else text_node.text or ""
                else:
                    value_node = cell.find("a:v", ns)
                    value = "" if value_node is None else value_node.text or ""
                    if cell_type == "s" and value != "":
                        value = shared_strings[int(value)]

                values[idx] = value

            rows.append([values.get(i, "") for i in range(max_idx + 1)])

    if not rows:
        return pd.DataFrame()

    header = rows[0]
    records = []
    for row in rows[1:]:
        padded = row + [""] * max(0, len(header) - len(row))
        records.append(
            {
                header[i]: padded[i] if i < len(padded) else ""
                for i in range(len(header))
            }
        )

    return pd.DataFrame(records)


In [2]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError("Tidak ada file yang ditemukan dari kandidat: " + ", ".join(map(str, paths)))


CHECKPOINT_PATH = first_existing(
    [
        PACKAGE_DIR / "outputs" / "best_intermediate_fusion.pt",
        PROJECT_ROOT / "ouputmodel" / "best_intermediate_fusion.pt",
    ]
)
METADATA_PATH = first_existing(
    [
        PACKAGE_DIR / "outputs" / "best_intermediate_fusion_metadata.json",
        PROJECT_ROOT / "ouputmodel" / "best_intermediate_fusion_metadata.json",
    ]
)

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
id_to_label = {int(key): value for key, value in metadata["id_to_label"].items()}
label_names = [id_to_label[index] for index in sorted(id_to_label)]

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Metadata: {METADATA_PATH}")
print(f"Best epoch: {metadata.get('best_epoch')}")
print(f"Best validation loss: {metadata.get('best_val_loss')}")
print(f"Labels: {label_names}")


Checkpoint: C:\Users\elsae\.vscode\Documents\city-branding-multimodal\multimodal_citybranding\outputs\best_intermediate_fusion.pt
Metadata: C:\Users\elsae\.vscode\Documents\city-branding-multimodal\multimodal_citybranding\outputs\best_intermediate_fusion_metadata.json
Best epoch: 7
Best validation loss: 0.08443489050988456
Labels: ['Accessibility', 'Amenities', 'Ancillary', 'Attraction']


In [3]:
DEVICE = resolve_device("auto")
BATCH_SIZE = int(metadata.get("batch_size", 16))
TEXT_MAX_LENGTH = int(metadata.get("text_max_length", 160))

feature_extractor = MultimodalFeatureExtractor(
    clip_model_name=metadata["clip_model_name"],
    text_model_name=metadata["text_model_name"],
    device=DEVICE,
)

model = IntermediateFusionClassifier(
    image_dim=int(metadata["image_dim"]),
    text_dim=int(metadata["text_dim"]),
    project_dim=int(metadata["project_dim"]),
    num_classes=int(metadata["num_classes"]),
).to(DEVICE)

try:
    state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model.load_state_dict(state_dict)
model.eval()

print(f"Device: {DEVICE}")
print("Model checkpoint terbaik berhasil dimuat.")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cpu
Model checkpoint terbaik berhasil dimuat.


In [4]:
def resolve_image_path(value):
    path = Path(str(value)).expanduser()
    if path.is_absolute() and path.exists():
        return path

    candidates = [
        PROJECT_ROOT / path,
        DEFAULT_IMAGE_DIR / path,
        PACKAGE_DIR / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Image tidak ditemukan: {value}")


def predict_batch(data, image_col="image_path", caption_col="caption", batch_size=BATCH_SIZE):
    df = pd.DataFrame(data).copy()

    if image_col not in df.columns and "File" in df.columns:
        image_col = "File"
    if caption_col not in df.columns and "Caption" in df.columns:
        caption_col = "Caption"

    if image_col not in df.columns:
        raise KeyError(f"Kolom image tidak ditemukan: {image_col}")
    if caption_col not in df.columns:
        raise KeyError(f"Kolom caption tidak ditemukan: {caption_col}")

    image_paths = [resolve_image_path(value) for value in df[image_col].tolist()]
    captions = df[caption_col].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip().tolist()

    image_features = feature_extractor.extract_image_features(
        image_paths,
        augment=False,
        batch_size=batch_size,
        desc="inference image features",
    )
    text_features = feature_extractor.extract_text_features(
        captions,
        batch_size=batch_size,
        max_length=TEXT_MAX_LENGTH,
        desc="inference text features",
    )

    probabilities = []
    with torch.no_grad():
        for start in range(0, len(df), batch_size):
            image_batch = image_features[start : start + batch_size].to(DEVICE)
            text_batch = text_features[start : start + batch_size].to(DEVICE)
            logits = model(image_batch, text_batch)
            probabilities.append(F.softmax(logits, dim=1).cpu())

    probabilities = torch.cat(probabilities, dim=0).numpy()
    pred_ids = probabilities.argmax(axis=1)
    pred_probs = probabilities.max(axis=1)

    result = df.copy()
    result["label"] = [id_to_label[int(index)] for index in pred_ids]
    result["probability"] = pred_probs

    for class_index, label in enumerate(label_names):
        result[f"prob_{label}"] = probabilities[:, class_index]

    return result


def predict_one(image_path, caption):
    result = predict_batch(
        [{"image_path": image_path, "caption": caption}],
        image_col="image_path",
        caption_col="caption",
    )
    probability_columns = [f"prob_{label}" for label in label_names]
    return result[["label", "probability", *probability_columns]]


## Inference Satu Data

Ubah `INPUT_IMAGE_PATH` dan `INPUT_CAPTION` sesuai data yang ingin diprediksi. Contoh di bawah memakai baris pertama dari dataset skema 2.


In [5]:
sample_df = read_xlsx_basic(DEFAULT_LABEL_FILE)
sample_df = sample_df[sample_df["File"].fillna("").astype(str).str.strip().ne("")].reset_index(drop=True)

INPUT_IMAGE_PATH = DEFAULT_IMAGE_DIR / sample_df.loc[0, "File"]
INPUT_CAPTION = sample_df.loc[0, "Caption"]

predict_one(INPUT_IMAGE_PATH, INPUT_CAPTION)


inference image features:   0%|          | 0/1 [00:00<?, ?it/s]

inference text features:   0%|          | 0/1 [00:00<?, ?it/s]

,label,probability,prob_Accessibility,prob_Amenities,prob_Ancillary,prob_Attraction
0,Attraction,0.849782,0.056105,0.075847,0.018266,0.849782


## Inference Batch

Gunakan `predict_batch` untuk banyak data sekaligus. Output utamanya adalah `label` dan `probability`; kolom `prob_<nama_label>` berisi probabilitas tiap kelas.


In [6]:
batch_input = sample_df[["File", "Caption"]].head(5).copy()
batch_predictions = predict_batch(batch_input, image_col="File", caption_col="Caption")

probability_columns = [f"prob_{label}" for label in label_names]
batch_predictions[["File", "label", "probability", *probability_columns]]


inference image features:   0%|          | 0/1 [00:00<?, ?it/s]

inference text features:   0%|          | 0/1 [00:00<?, ?it/s]

,File,label,probability,prob_Accessibility,prob_Amenities,prob_Ancillary,prob_Attraction
0,1.jpg,Attraction,0.849782,0.056105,0.075847,0.018266,0.849782
1,2.jpg,Attraction,0.821220,0.020555,0.143533,0.014692,0.821220
2,3.jpg,Amenities,0.721480,0.013410,0.721480,0.013059,0.252052
3,4.jpg,Attraction,0.687263,0.054593,0.246892,0.011253,0.687263
4,5.jpg,Ancillary,0.877322,0.052618,0.016342,0.877322,0.053718
